# 3. Document Reality & Ingestion with Docling


In this lab, all of the customer data are in “native PDFs,” meaning we will not be using the OCR capabilities. This is done to simplify the labs and speed up the ingest process. 

Purpose:
>Expose why documents break naïve AI approaches and demonstrate how a repeatable ingestion pipeline (Docling) turns fragile inputs into reliable data.

## 3.1  The Reality: Documents Are Not Data


Before we touch any AI system, we need to confront a simple truth:

* PDFs are presentation formats
* They optimize for human reading, not machine reasoning
* Structure is often implicit, inconsistent, or lost entirely
Common failure modes you will see in real customer documents:
* Structure loss
    * Tables flattened into paragraphs
    * Headings merged into body text
    * Lists lose ordering and semantics
* OCR noise
    * Random line breaks
    * Broken characters
    * Hyphenation artifacts
    * “O” vs “0” errors
* Ambiguity
    * Multiple document versions
    * Conflicting definitions
    * Domain assumptions never stated explicitly
>“Most AI failures don’t start at the model — they start at ingestion.”

## 3.2 Quick Exercise: Spot the Failure
Open the source PDF file `Basic-Fantasy-RPG-Rules-r142.pdf`  and inspect 2–3 representative pages and answer:

* Which parts of this document would be hardest to extract correctly?
* Which parts matter most for correctness?
* If an AI answer were wrong, where would you look first?

Facilitator note:

>Keep this observational. No tools yet. The goal is to build discomfort, not solve anything.


## 3.3 Why Ingestion Is the First Real Technical Step 

The source document has images, tables, and figures. All of these are displayed in a multi-columnar layout. While there are many ways to address these, Docling does an excellent job at handling documents with complex formatting.

## Introducing Docling: What It Solves

Docling is a document-to-structured-data pipeline. Its job isn’t to be clever. It’s not trying to interpret intent or answer questions. It takes messy, real-world documents — PDFs, reports, forms, multi-column layouts, tables that were clearly designed by committee — and turns them into structured, predictable data.

And the key word there is predictable.

Docling is deterministic and repeatable. Given the same input, you get the same structured output. That might sound unremarkable, but in enterprise systems, repeatability is everything. If your downstream AI workflow depends on document structure, you can’t afford to have the shape of that data drift unpredictably from run to run.

It’s also built specifically for enterprise document messiness. Not the clean demo PDFs. The real ones. The ones with embedded tables, inconsistent headers, strange formatting, legacy templates, scanned inserts — the artifacts of years of operational reality.

Now, just as important is what Docling is not.

Docling is not an LLM. It’s not generating text, reasoning over content, or improvising structure. It’s not a retrieval system. It doesn’t index and search across a knowledge base. And it’s definitely not a one-off PDF conversion script that someone wrote during a hackathon and hopes never breaks.

If we think about modern AI systems as a pipeline, Docling is the contract layer. It gives us a stable, structured representation of documents before we hand anything to downstream AI — whether that’s embeddings, retrieval, classification, or LLM orchestration.
And that framing matters.

Because what Docling really gives us is a contract between documents and downstream AI.

It says: “No matter how messy the input is, here is the shape of the data you can rely on.”

That contract is what allows the rest of the system to scale.


### 3.4.1 Install Docling

Run the following to install docling.

> **Note:** You may see yellow `DEPRECATION` warnings about `pylatexenc` and `antlr4-python3-runtime` using a legacy build mechanism, and red `ERROR` messages about dependency conflicts for `urllib3` and `dill`. These are safe to ignore. The warnings come from upstream packages that have not yet adopted the newer Python packaging standard, and the dependency conflicts involve packages (`appengine-python-standard`, `feast`) that are not used in this workshop. Docling will install and function correctly.

In [1]:
# ! pip install docling -q

! pip install docling==2.58.0 pypdfium2==4.30.0 -q

  DEPRECATION: Building 'pylatexenc' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pylatexenc'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  DEPRECATION: Building 'antlr4-python3-runtime' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'antlr4-python3-runtime'. Discussion can be found at https://github.com/pypa/pip/issues/6334


>Note: A full deep dive into Docling is outside the scope of this lab. More information is available at www.docling.ai 

### 3.4.2 Convert a PDF to Markdown

Before we can continue, we need clean text. Most document formats, PDFs included, were designed for visual layout, not semantic structure: headings, paragraphs, and lists are just positioned elements on a page. 

For RAG systems, that layout-first format creates noise. Converting a PDF to Markdown restores useful structure: headings become explicit, paragraphs are clearer, and the document becomes much easier to split intelligently later.

>Note: The following code skips OCR for the sake of time. This assumes the PDF already contains extractable text rather than scanned images. In a production system with varied kinds of source documents, you would typically include an OCR step to handle scanned documents and ensure consistent text extraction.

> **Note:** You may see warning messages when the conversion runs. These are harmless and can be ignored:
> - **`TqdmWarning: IProgress not found`** — This is a cosmetic Jupyter widget warning. The progress bars will still work; they just render as plain text instead of graphical widgets.
> - **`UserWarning: CUDA initialization` / `Can't initialize NVML`** — This means no compatible NVIDIA GPU was detected (or the driver is too old). Docling will automatically fall back to CPU processing and complete the conversion normally.

In [ ]:
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption

source = "../docs/Basic-Fantasy-RPG-Rules-r142.pdf"

# Create PDF pipeline options
pdf_options = PdfPipelineOptions()
pdf_options.do_ocr = False  # Disable OCR

# Configure converter with format options
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_options)
    }
)

import time

start = time.time()

doc = converter.convert(source).document

markdown_content = doc.export_to_markdown()
output_path = "../docs/Basic-Fantasy-RPG-Rules-r142.md"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(markdown_content)

elapsed = time.time() - start
print(f"Saved to {output_path}")
print(f"Ingestion completed in {elapsed:.1f} seconds")

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/app-root/lib64/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


> **Took too long?** If the conversion above is taking more than a few minutes on a CPU-only environment, you can interrupt the cell and use the prebuilt version instead. Run the cell below to copy it into place.

In [ ]:
import shutil, os

prebuilt_md = "../prebuilt/Basic-Fantasy-RPG-Rules-r142.md"
output_path = "../docs/Basic-Fantasy-RPG-Rules-r142.md"

if not os.path.exists(output_path):
    shutil.copy(prebuilt_md, output_path)
    print(f"Copied prebuilt markdown to {output_path}")
else:
    print(f"{output_path} already exists — no action needed.")

On a GPU-equipped workbench (NVIDIA L40S, 16 GiB RAM), this conversion completed in approximately 135 seconds for a 106 MB PDF. 

Your time may vary depending on document complexity, hardware profile, and whether OCR is enabled. 

This is a one-time cost per document. Once the markdown is generated, it can be reused across all downstream steps without re-running the conversion.


### 3.5.1 Inspect the Docling Output 

Open the newly created `../docs/Basic-Fantasy-RPG-Rules-r142.md` file and examine it.   (You may have to refresh the directory to view the file.) 


As you scroll down, you will notice that the semantic structure of the document has been largely preserved. 

There's a table at line 238 that faithfully represents the table on page 4 in the source document.

![Source Document Page](images/page.png)



```

| Ability Score   |   Bonus/Penalty |
|-----------------|-----------------|
| 3               |              -3 |
| 4-5             |              -2 |
| 6-8             |              -1 |
| 9-12            |               0 |
| 13-15           |              +1 |
| 16-17           |              +2 |
| 18              |              +3 |

```


The model hasn't "understood" anything yet. Nothing new was added. No intelligence was injected. But something important happened: structure was preserved, and ambiguity was reduced.

There are many ways to build ingestion pipelines, and they vary widely in how faithfully they preserve source material. When structure is lost during ingestion, every downstream task inherits that degradation. "Garbage in, garbage out." still applies in the AI era.

Docling's value is not that it makes documents smarter. It ensures they arrive intact.